# Reprodução de Figuras e Indicadores Científicos do Atlas Histórico

Este notebook reproduz as análises quantitativas, matrizes de cobertura e estados do mundo cartográficos da pesquisa:
- **Projeto:** Atlas Histórico da Criminalidade no Rio de Janeiro (1950–2026)
- **Autoria:** Daniel Farias
- **Princípio:** Auditabilidade, transparência empírica e reprodutibilidade (Zero IA inventada).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Adiciona a raiz do projeto ao path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.database import SessionLocal
from app.services.atlas_service import AtlasService
from app.services.coverage_service import CoverageService
from app.services.ethics_guard import EthicsGuard

print("✓ Módulos do Atlas carregados com sucesso!")

## 1. Matriz de Cobertura Documental e Diagnóstico de Lacunas

In [ ]:
cov_service = CoverageService()
summary = cov_service.get_coverage_summary(is_demo=False)

print(f"Total de pares (região, década) analisados: {summary['total_cells']}")
print(f"Densidade Média de Evidências: {summary['overall_evidence_density']}")
print("\nDistribuição por Status de Cobertura:")
for status, count in summary['counts'].items():
    pct = summary['percentages'][status]
    print(f"  - {status:20s}: {count:3d} células ({pct:5.1f}%)")

In [ ]:
# Médias de densidade documental por década
df_decadas = pd.DataFrame(list(summary['decade_averages'].items()), columns=['Década', 'Densidade de Evidência'])
df_decadas['Década'] = df_decadas['Década'].astype(str) + 's'
df_decadas

## 2. Inspeção do Estado do Mundo (WorldState) para 1979

O ano de 1979 marca o massacre do presídio da Ilha Grande e a gênese da Falange Vermelha.

In [ ]:
atlas_service = AtlasService()
ws_1979 = atlas_service.get_world_state(year=1979, is_demo=False)

print("Metadados de Auditoria do Estado de 1979:")
for k, v in ws_1979.metadata.items():
    print(f"  {k}: {v}")

print(f"\nTotal de Territórios Documentados: {len(ws_1979.territories)}")
print(f"Total de Equipamentos Institucionais Ativos: {len(ws_1979.facilities)}")
print(f"Total de Eventos Ocorridos em 1979: {len(ws_1979.events)}")

## 3. Rastreabilidade Epistemológica (Pixel ao Trecho Literal)

In [ ]:
if ws_1979.territories:
    feat_id = ws_1979.territories[0]['properties']['feature_id']
    rec = atlas_service.get_epistemological_record(feat_id)
    print(f"Ficha do Objeto: {rec.feature_id}")
    print(f"Título: {rec.title}")
    print(f"Relação Territorial: {rec.relation_type}")
    print(f"Organização: {rec.actor_name} ({rec.actor_acronym})")
    print(f"Força da Evidência: {rec.evidence_strength}")
    print(f"Dataset Cartográfico: {rec.dataset_provenance.get('dataset_name') if rec.dataset_provenance else 'N/D'}")
    print(f"Aviso de Anacronismo: {rec.anachronism_warning}")

## 4. Verificação do Embargo Ético de 24 Meses (Ano Recente: 2026)

In [ ]:
ws_2026 = atlas_service.get_world_state(year=2026, is_demo=False)
print(f"Embargo Ativo em 2026: {ws_2026.metadata.get('ethics_embargo_active')}")
print(f"Justificativa Ética: {ws_2026.metadata.get('ethics_embargo_reason')}")

for ev in ws_2026.events:
    print(f"- Evento: {ev['properties']['title']}")
    print(f"  Coordenadas Agregadas: {ev['geometry']['coordinates']}")
    print(f"  Precisão Espacial: {ev['properties']['location_precision']}")